<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/Day8/regression_medicalexpenses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df = pd.read_csv('/content/medical expenses.csv')

In [2]:
df.shape

(1338, 7)

In [3]:
df.isna().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
expenses,0


In [4]:
df.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'expenses'], dtype='object')

In [5]:
df.head()

,age,sex,bmi,children,smoker,region,expenses
0,19,female,27.9,0,yes,southwest,16884.92
1,18,male,33.8,1,no,southeast,1725.55
2,28,male,33.0,3,no,southeast,4449.46
3,33,male,22.7,0,no,northwest,21984.47
4,32,male,28.9,0,no,northwest,3866.86


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   expenses  1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [7]:
df.drop_duplicates(inplace=True)
df.shape

(1337, 7)

In [8]:
df.nunique()

,0
age,47
sex,2
bmi,275
children,6
smoker,2
region,4
expenses,1337


In [9]:
cont_col = ['age','bmi']

In [10]:
df_ohe = pd.get_dummies(df)
df_ohe.shape

(1337, 12)

### In case of Categorical target column do one hot encoding after Separating X & Y

## Separate X & Y
- Target col :: expenses

In [11]:
X = df_ohe.drop('expenses',axis=1)
y = df_ohe['expenses']
X.shape,y.shape

((1337, 11), (1337,))

## Target column in regression can be transformed using power transforms , in case it is highly skewed (>3), In this case when we do the predictions we have to do inverse transform, for eg: If we apply log transform inverse transform is e^x

## Performance of the model in regression in is better if input continuous features are normally distributed

## So in case input continuous column have high skewness it's good to apply power transforms on them

In [13]:
X.skew()

,0
age,0.054781
bmi,0.284463
children,0.937421
sex_female,0.019469
sex_male,-0.019469
smoker_no,-1.463601
smoker_yes,1.463601
region_northeast,1.204009
region_northwest,1.204009
region_southeast,1.024467


In [14]:
y.skew()

np.float64(1.5153909165486397)

In [16]:
# No need to apply stratify as it is regression Problem
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=7)
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((935, 11), (402, 11), (935,), (402,))

## Standardize Cont Columns

In [17]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train[cont_col] = sc.fit_transform(X_train[cont_col])
X_test[cont_col] = sc.transform(X_test[cont_col])

In [18]:
X_train[cont_col].mean()

,0
age,8.359326e-17
bmi,2.659786e-16


In [22]:
# Apply Linear Regression (Uses Ordinary Least Square Method)
# Applied Multiple Linear regression as we have taken input from multiple features
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error,mean_absolute_error,r2_score
lr = LinearRegression()
lr.fit(X_train,y_train)
y_pred = lr.predict(X_test)

In [21]:
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  6210.2716157612385
MAE ::  4262.336281450989
R2 Score ::  0.7324133376864894


### Y= mx+c

In [25]:
# b1 , b2, b3
lr.coef_

array([  3691.76308759,   2141.74157658,    550.95853205,     77.40983229,
          -77.40983229, -11932.614766  ,  11932.614766  ,    807.07943605,
           24.93368793,   -638.37864692,   -193.63447706])

In [26]:
# Y intercept --> b0
lr.intercept_

np.float64(19626.04536523765)

## Polynomial Regression

- Step 1 : Add polynomial features to the data
- Step 2 : Apply Linear Regression

In [29]:
from sklearn.preprocessing import PolynomialFeatures
pf = PolynomialFeatures(degree=2)
X_train_pf = pf.fit_transform(X_train)
X_test_pf = pf.transform(X_test)
X_train_pf.shape,X_test_pf.shape

((935, 78), (402, 78))

In [33]:
lr = LinearRegression()
lr.fit(X_train_pf,y_train)
y_pred = lr.predict(X_test_pf)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5164.675423679967
MAE ::  2963.5756733421345
R2 Score ::  0.8149328492092578


In [35]:
# Lasso for Degree 2
from sklearn.linear_model import Ridge,Lasso
ls = Lasso(alpha=1, random_state=7)
ls.fit(X_train_pf,y_train)
y_pred = ls.predict(X_test_pf)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5163.539814272533
MAE ::  2963.630309903382
R2 Score ::  0.8150142254332273


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.626e+08, tolerance: 1.380e+07
  model = cd_fast.enet_coordinate_descent(


In [36]:
# Ridge For Degree 2
rd = Ridge(alpha=1, random_state=7)
rd.fit(X_train_pf,y_train)
y_pred = rd.predict(X_test_pf)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5161.851912428195
MAE ::  2965.9433329455424
R2 Score ::  0.8151351451153899


In [32]:
# Exponentially Increasing the Number of Dimension by Increasing Degree
from sklearn.preprocessing import PolynomialFeatures
pf3 = PolynomialFeatures(degree=3)
X_train_pf3 = pf3.fit_transform(X_train)
X_test_pf3 = pf3.transform(X_test)
X_train_pf3.shape,X_test_pf3.shape

((935, 364), (402, 364))

In [34]:
lr = LinearRegression()
lr.fit(X_train_pf3,y_train)
y_pred = lr.predict(X_test_pf3)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5261.807789576539
MAE ::  3122.1429446805687
R2 Score ::  0.8079062515325234


In [39]:
# lasso for Degree 3
ls = Lasso(alpha=1, random_state=7)
ls.fit(X_train_pf3,y_train)
y_pred = ls.predict(X_test_pf3)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5252.85949583102
MAE ::  3111.0686949540604
R2 Score ::  0.8085590498695285


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.758e+08, tolerance: 1.380e+07
  model = cd_fast.enet_coordinate_descent(


In [40]:
# Ridge for degree 3
rd = Ridge(alpha=1, random_state=7)
rd.fit(X_train_pf3,y_train)
y_pred = rd.predict(X_test_pf3)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5253.060993101152
MAE ::  3116.5005623617185
R2 Score ::  0.8085443624145414


## Fine Tune Alpha

### Lasso

default alpha = 1

1. Use high alpha :: 100

2. Use Low alpha :: 0.5

In [41]:
# lasso for Degree 3
ls = Lasso(alpha=100, random_state=7)
ls.fit(X_train_pf3,y_train)
y_pred = ls.predict(X_test_pf3)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5071.194290856006
MAE ::  3091.773647293133
R2 Score ::  0.8215716859419894


In [44]:
(ls.coef_ !=0).sum(),(ls.coef_ ==0).sum()

(np.int64(33), np.int64(331))

In [45]:
# lasso for Degree 3
ls = Lasso(alpha=0.5, random_state=7)
ls.fit(X_train_pf3,y_train)
y_pred = ls.predict(X_test_pf3)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5257.653477590329
MAE ::  3117.3549077340717
R2 Score ::  0.8082094561981368


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.475e+09, tolerance: 1.380e+07
  model = cd_fast.enet_coordinate_descent(


In [46]:
(ls.coef_ !=0).sum(),(ls.coef_ ==0).sum()

(np.int64(147), np.int64(217))

### Ridge

- High Alpha :: 5

- Low Alpha :: 0.5

In [51]:
# Ridge for degree 3
rd = Ridge(alpha=50, random_state=7)
rd.fit(X_train_pf3,y_train)
y_pred = rd.predict(X_test_pf3)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5140.727855249266
MAE ::  3153.0285780720264
R2 Score ::  0.8166451091126958


In [58]:
rd.coef_

array([ 0.00000000e+00,  1.07999938e+03,  1.49355547e+03,  1.72424943e+01,
       -6.40047928e+00,  6.40047928e+00, -2.49798050e+03,  2.49798050e+03,
        2.41383122e+02, -1.08399609e+02, -6.75447254e+01, -6.54387871e+01,
        4.39817304e+01,  3.38498382e+01, -2.27208096e+02,  4.06075443e+02,
        6.73923932e+02,  5.53958951e+02,  5.26040424e+02, -2.35506552e+00,
        2.75347240e+02,  5.34573167e+02,  2.72434033e+02, -1.89340623e+02,
       -1.22377354e+01,  7.66503986e+02,  7.27051485e+02, -1.22737102e+03,
        2.72092649e+03,  1.89721983e+02,  7.73706745e+02, -2.05552509e+01,
        5.50681993e+02,  7.13361088e+00,  9.57283749e+01, -7.84858807e+01,
        4.03681092e+02, -3.86438598e+02,  2.15525904e+02,  3.99095939e+02,
       -3.45543447e+02, -2.51835902e+02, -6.40047928e+00,  0.00000000e+00,
       -1.22943017e+03,  1.22302969e+03,  1.05461283e+02,  6.60946392e+01,
       -1.23608589e+02, -5.43478120e+01,  6.40047928e+00, -1.26855033e+03,
        1.27495081e+03,  

In [59]:
(rd.coef_ == 0).sum()

np.int64(89)

In [55]:
# Ridge for degree 3
rd = Ridge(alpha=0.5, random_state=7)
rd.fit(X_train_pf3,y_train)
y_pred = rd.predict(X_test_pf3)
print('RMSE :: ',root_mean_squared_error(y_test,y_pred))
print('MAE :: ',mean_absolute_error(y_test,y_pred))
print('R2 Score :: ',r2_score(y_test,y_pred))

RMSE ::  5257.359271332407
MAE ::  3119.2919076773874
R2 Score ::  0.8082309199175329


In [56]:
rd.coef_

array([ 0.00000000e+00,  1.07999938e+03,  1.49355547e+03,  1.72424943e+01,
       -6.40047928e+00,  6.40047928e+00, -2.49798050e+03,  2.49798050e+03,
        2.41383122e+02, -1.08399609e+02, -6.75447254e+01, -6.54387871e+01,
        4.39817304e+01,  3.38498382e+01, -2.27208096e+02,  4.06075443e+02,
        6.73923932e+02,  5.53958951e+02,  5.26040424e+02, -2.35506552e+00,
        2.75347240e+02,  5.34573167e+02,  2.72434033e+02, -1.89340623e+02,
       -1.22377354e+01,  7.66503986e+02,  7.27051485e+02, -1.22737102e+03,
        2.72092649e+03,  1.89721983e+02,  7.73706745e+02, -2.05552509e+01,
        5.50681993e+02,  7.13361088e+00,  9.57283749e+01, -7.84858807e+01,
        4.03681092e+02, -3.86438598e+02,  2.15525904e+02,  3.99095939e+02,
       -3.45543447e+02, -2.51835902e+02, -6.40047928e+00,  0.00000000e+00,
       -1.22943017e+03,  1.22302969e+03,  1.05461283e+02,  6.60946392e+01,
       -1.23608589e+02, -5.43478120e+01,  6.40047928e+00, -1.26855033e+03,
        1.27495081e+03,  

In [57]:
(rd.coef_ == 0).sum() ## Only Making the value reduce not making it zero

np.int64(89)